In [2]:
import os
print(os.listdir('/kaggle/input'))

!pip install -q transformers datasets sentencepiece accelerate

import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU count:", torch.cuda.device_count())
print("GPU memory allocated:", torch.cuda.memory_allocated() / 1e9, "GB")

['datasets']
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 82.5 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires numba<0.62.0,>=0.60.0, but you have numba 0.65.1 which is incompatible.
cuml-cu12 26.2.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cudf-cu12 26.2.1 requires numba<0.62.0,>=0.60.0, but you have numba

In [3]:
import pandas as pd
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import Dataset

data_dir = "/kaggle/input/datasets/henrietteutatsineza/dataset"
train = pd.read_csv(f"{data_dir}/Train.csv")
test = pd.read_csv(f"{data_dir}/Test.csv")

train['input_text'] = "answer in " + train['subset'] + ": " + train['input']
test['input_text'] = "answer in " + test['subset'] + ": " + test['input']

train_df, val_df = train_test_split(train, test_size=0.1, random_state=42, stratify=train['subset'])
print("Train size:", len(train_df), "| Val size:", len(val_df))

model_name = "google/mt5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

train_ds = Dataset.from_pandas(train_df[['input_text', 'output']].reset_index(drop=True))
val_ds = Dataset.from_pandas(val_df[['input_text', 'output']].reset_index(drop=True))

max_input_len = 128
max_output_len = 128

def preprocess(examples):
    model_inputs = tokenizer(examples['input_text'], max_length=max_input_len, truncation=True, padding='max_length')
    labels = tokenizer(examples['output'], max_length=max_output_len, truncation=True, padding='max_length')
    labels['input_ids'] = [[(t if t != tokenizer.pad_token_id else -100) for t in label] for label in labels['input_ids']]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_tokenized = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tokenized = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)

print("Tokenization done")

Train size: 26833 | Val size: 2982


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Map:   0%|          | 0/26833 [00:00<?, ? examples/s]

Map:   0%|          | 0/2982 [00:00<?, ? examples/s]

Tokenization done


In [4]:
import warnings
import logging
warnings.filterwarnings("ignore")
logging.set_verbosity_error()
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)

training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/mt5-small-healthqa-v2",
    eval_strategy="steps",
    eval_steps=1000,
    save_strategy="steps",
    save_steps=1000,
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=4,
    predict_with_generate=True,
    fp16=False,
    bf16=False,
    logging_steps=100,
    save_total_limit=1,
    load_best_model_at_end=False,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=train_tokenized,
    eval_dataset=val_tokenized,
    data_collator=data_collator,
    processing_class=tokenizer,
)

print("Trainer set up (batch=4, 4 epochs, fp32). Starting training...")
trainer.train()

Trainer set up (batch=4, 4 epochs, fp32). Starting training...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss
1000,6.386992,5.224747
2000,5.616499,4.646133
3000,5.143514,4.344485
4000,4.844348,4.119085
5000,4.684122,3.966040
6000,4.558967,3.833289
7000,4.363629,3.743076
8000,4.321959,3.662214
9000,4.321990,3.593009
10000,4.126975,3.546198


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


...

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=13420, training_loss=4.841189686623132, metrics={'train_runtime': 14420.0147, 'train_samples_per_second': 7.443, 'train_steps_per_second': 0.931, 'total_flos': 1.418794045538304e+16, 'train_loss': 4.841189686623132, 'epoch': 4.0})

In [5]:
import torch
import pandas as pd

def generate_predictions(model, tokenizer, texts, max_length=128, batch_size=8):
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    predictions = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                          truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=max_length, num_beams=4)
        predictions.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
        if i % 100 == 0:
            print(f"Processed {i}/{len(texts)}")
    return predictions

# Generate on test set
test_texts = test['input_text'].tolist()
print("Generating predictions on test set...")
test_preds = generate_predictions(model, tokenizer, test_texts)

# Build submission
submission = pd.DataFrame({
    'ID': test['ID'],
    'TargetRLF1': test_preds,
    'TargetR1F1': test_preds,
    'TargetLLM': test_preds
})

submission.to_csv('/kaggle/working/submission_exp2.csv', index=False)
print("Submission saved!")
print(submission.head())
print("Shape:", submission.shape)

Generating predictions on test set...
Processed 0/2618
Processed 200/2618
Processed 400/2618
Processed 600/2618
Processed 800/2618
Processed 1000/2618
Processed 1200/2618
Processed 1400/2618
Processed 1600/2618
Processed 1800/2618
Processed 2000/2618
Processed 2200/2618
Processed 2400/2618
Processed 2600/2618
Submission saved!
                       ID                                         TargetRLF1  \
0  ID_TS_Aka_Gha_A3B1799D  Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumay...   
1  ID_TS_Aka_Gha_1C80317F  Mmabun betumi afi hokwan a mmabun wɔ sɛ wonya ...   
2  ID_TS_Aka_Gha_06671AD1  Mmabun bɛtumi afa so ehunu nsusuanso a ɛtumi a...   
3  ID_TS_Aka_Gha_BDD640FB  Amammerɛ mu mmra, asetena mu suban, ne tumi mu...   
4  ID_TS_Aka_Gha_46685257  Mmara nsesaeɛ 'policies advocacy' ho hia ma mm...   

                                          TargetR1F1  \
0  Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumay...   
1  Mmabun betumi afi hokwan a mmabun wɔ sɛ wonya ...   
2  Mmabun bɛtumi afa s

In [2]:
from IPython.display import FileLink
FileLink('/kaggle/working/submission_exp2.csv')

/kaggle/working/submission_exp2.csv

In [3]:
import torch
import pandas as pd

def generate_predictions(model, tokenizer, texts, max_length=128, batch_size=8):
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    predictions = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True,
                          truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=max_length, num_beams=4)
        predictions.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
        if i % 500 == 0:
            print(f"Processed {i}/{len(texts)}")
    return predictions

test_texts = test['input_text'].tolist()
print("Generating predictions...")
test_preds = generate_predictions(model, tokenizer, test_texts)

submission = pd.DataFrame({
    'ID': test['ID'],
    'TargetRLF1': test_preds,
    'TargetR1F1': test_preds,
    'TargetLLM': test_preds
})

submission.to_csv('/kaggle/working/submission_exp2.csv', index=False)
print("Done! Downloading...")

from IPython.display import FileLink
FileLink('/kaggle/working/submission_exp2.csv')

NameError: name 'test' is not defined

In [5]:
import os
for root, dirs, files in os.walk('/kaggle/working'):
    for f in files:
        print(os.path.join(root, f))

/kaggle/working/.virtual_documents/__notebook_source__.ipynb


In [6]:
import os, pandas as pd, torch
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer, DataCollatorForSeq2Seq
from datasets import Dataset

# ── Data ──────────────────────────────────────────────────────────────
import glob
data_dir = os.path.dirname(glob.glob('/kaggle/input/**/Train.csv', recursive=True)[0])
train = pd.read_csv(f"{data_dir}/Train.csv")
test  = pd.read_csv(f"{data_dir}/Test.csv")

train['input_text'] = "answer in " + train['subset'] + ": " + train['input']
test['input_text']  = "answer in " + test['subset']  + ": " + test['input']

train_df, val_df = train_test_split(train, test_size=0.1, random_state=42, stratify=train['subset'])
print(f"Train: {len(train_df)} | Val: {len(val_df)}")

# ── Model ─────────────────────────────────────────────────────────────
model_name = "google/mt5-small"
tokenizer  = AutoTokenizer.from_pretrained(model_name)
model      = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# ── Tokenize ──────────────────────────────────────────────────────────
def preprocess(examples):
    model_inputs = tokenizer(examples['input_text'], max_length=128, truncation=True, padding='max_length')
    labels = tokenizer(examples['output'], max_length=128, truncation=True, padding='max_length')
    labels['input_ids'] = [[(t if t != tokenizer.pad_token_id else -100) for t in l] for l in labels['input_ids']]
    model_inputs['labels'] = labels['input_ids']
    return model_inputs

train_ds = Dataset.from_pandas(train_df[['input_text','output']].reset_index(drop=True))
val_ds   = Dataset.from_pandas(val_df[['input_text','output']].reset_index(drop=True))
train_tokenized = train_ds.map(preprocess, batched=True, remove_columns=train_ds.column_names)
val_tokenized   = val_ds.map(preprocess, batched=True, remove_columns=val_ds.column_names)
print("Tokenization done")

# ── Train ─────────────────────────────────────────────────────────────
data_collator = DataCollatorForSeq2Seq(tokenizer=tokenizer, model=model)
training_args = Seq2SeqTrainingArguments(
    output_dir="/kaggle/working/mt5-small-v2",
    eval_strategy="steps", eval_steps=1000,
    save_strategy="no",
    learning_rate=3e-4,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    num_train_epochs=4,
    predict_with_generate=True,
    fp16=False, bf16=False,
    logging_steps=100,
    report_to="none"
)
trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=train_tokenized, eval_dataset=val_tokenized,
    data_collator=data_collator, processing_class=tokenizer,
)
print("Training started...")
trainer.train()

# ── Generate & Save ───────────────────────────────────────────────────
def generate_predictions(model, tokenizer, texts, max_length=128, batch_size=8):
    model.eval()
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    preds = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        inputs = tokenizer(batch, return_tensors="pt", padding=True, truncation=True, max_length=128).to(device)
        with torch.no_grad():
            outputs = model.generate(**inputs, max_length=max_length, num_beams=4)
        preds.extend(tokenizer.batch_decode(outputs, skip_special_tokens=True))
        if i % 500 == 0:
            print(f"Generated {i}/{len(texts)}")
    return preds

print("Generating predictions...")
test_preds = generate_predictions(model, tokenizer, test['input_text'].tolist())

submission = pd.DataFrame({
    'ID': test['ID'],
    'TargetRLF1': test_preds,
    'TargetR1F1': test_preds,
    'TargetLLM':  test_preds
})
submission.to_csv('/kaggle/working/submission_exp2.csv', index=False)
print(f"Saved! Shape: {submission.shape}")
print(submission.head(3))

from IPython.display import FileLink
FileLink('/kaggle/working/submission_exp2.csv')

Train: 26833 | Val: 2982


config.json:   0%|          | 0.00/553 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/82.0 [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/4.31M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/99.0 [00:00<?, ?B/s]

...

Tokenization done
Training started...


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Step,Training Loss,Validation Loss
1000,6.386992,5.224747
2000,5.616499,4.646133
3000,5.143514,4.344485
4000,4.844348,4.119085
5000,4.684122,3.966040
6000,4.558967,3.833289
7000,4.363629,3.743076
8000,4.321959,3.662214
9000,4.321990,3.593009
10000,4.126975,3.546198


Generating predictions...
Generated 0/2618
Generated 1000/2618
Generated 2000/2618
Saved! Shape: (2618, 4)
                       ID                                         TargetRLF1  \
0  ID_TS_Aka_Gha_A3B1799D  Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumay...   
1  ID_TS_Aka_Gha_1C80317F  Mmabun betumi afi hokwan a mmabun wɔ sɛ wonya ...   
2  ID_TS_Aka_Gha_06671AD1  Mmabun bɛtumi afa so ehunu nsusuanso a ɛtumi a...   

                                          TargetR1F1  \
0  Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumay...   
1  Mmabun betumi afi hokwan a mmabun wɔ sɛ wonya ...   
2  Mmabun bɛtumi afa so ehunu nsusuanso a ɛtumi a...   

                                           TargetLLM  
0  Nneɛma a wɔde bɛyɛ nkyerɛkyerɛ nneɛma, adwumay...  
1  Mmabun betumi afi hokwan a mmabun wɔ sɛ wonya ...  
2  Mmabun bɛtumi afa so ehunu nsusuanso a ɛtumi a...  


/kaggle/working/submission_exp2.csv